# KLA Image Restoration — Colab GPU Training

Self-contained notebook for the KLA SEMICON India Hackathon 2026 denoising + super-resolution baseline.

It clones the repository, installs a CUDA PyTorch build, generates a **synthetic** stand-in dataset (the official KLA pairs are not redistributable), trains the residual U-Net, and evaluates against the bicubic baseline. Every metric is written to `results/experiments.csv`.

**Honesty note:** all numbers produced here are on synthetic data and are labelled synthetic. They do not describe official KLA performance. See `docs/EXTERNAL_RESOURCES.md`.

**Runtime:** set `Runtime → Change runtime type → GPU (T4)` before running.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repository

Set `REPO_URL` to your pushed GitHub remote. A public repo needs no token; for a private repo use `https://<TOKEN>@github.com/<owner>/<repo>.git`.

In [ ]:
import os
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/Veer-WebDev/kla-image-restoration.git')
REPO_DIR = 'kla-image-restoration'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git log --oneline -1

## 3. Install dependencies

Colab ships a CUDA PyTorch build already, so install the rest without disturbing it.

In [ ]:
!pip install -q 'numpy<2' 'Pillow>=10,<12' 'PyYAML>=6,<7' tqdm 'scikit-image>=0.22,<0.25' lpips 'pandas>=2,<3' 'matplotlib>=3.8,<4'
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 4. Generate the synthetic dataset

Uses the in-repo, licence-clean generator (`scripts/make_fixtures.py`): wafer motifs (grid / lines / traces / contacts) degraded by the project's own deterministic engine. Increase `--count` for a longer run.

In [ ]:
!python scripts/make_fixtures.py --out data/GT_synth --count 200 --size 256 --seed 1234 --scale 2
# make_fixtures writes GT/ and NoisyLR/ under the --out root.
!ls data/GT_synth/GT | head && echo '---' && ls data/GT_synth/NoisyLR | head

## 5. Train

Trains on the synthetic pairs. `train_mode=mixed` combines the stored NoisyLR pairs with fresh on-the-fly degradations. Validation reports the bicubic baseline on identical inputs every epoch.

In [ ]:
!python train.py \
  --gt-dir data/GT_synth/GT \
  --noisy-dir data/GT_synth/NoisyLR \
  --epochs 40 \
  --set train.batch_size=16 \
  --set data.samples_per_image=8 \
  --set experiment_id=colab_t4_synth

### Optional: enable the degradation-aware (FiLM) variant

The standout upgrade. Run as a separate experiment to compare against the baseline.

In [ ]:
# !python train.py \
#   --gt-dir data/GT_synth/GT --noisy-dir data/GT_synth/NoisyLR \
#   --epochs 40 --set train.batch_size=16 \
#   --set model.degradation_aware=true --set model.condition_dim=4 \
#   --set experiment_id=colab_t4_synth_film

## 6. Inspect the results ledger

In [ ]:
import pandas as pd
df = pd.read_csv('results/experiments.csv')
cols = ['experiment_id','status','epochs_completed','gpu_name','best_epoch','best_psnr','best_ssim','bicubic_psnr']
cols = [c for c in cols if c in df.columns]
df[cols].tail(10)

## 7. Run inference on a held-out image

In [ ]:
import glob
ckpts = sorted(glob.glob('runs/**/best.pth', recursive=True)) + sorted(glob.glob('runs/**/last.pth', recursive=True))
print('checkpoints:', ckpts)
CKPT = ckpts[0] if ckpts else ''
print('using', CKPT)

In [ ]:
!python inference.py --checkpoint "$CKPT" --input_dir data/GT_synth/NoisyLR --output_dir runs/_colab_restored --scale 2 || true
!ls runs/_colab_restored 2>/dev/null | head

## 8. Download artifacts

Persist the checkpoint and ledger row to your machine (or mount Drive and copy there).

In [ ]:
from google.colab import files  # noqa
import shutil, os
if CKPT and os.path.exists(CKPT):
    shutil.copy(CKPT, 'best_colab.pth')
    files.download('best_colab.pth')
files.download('results/experiments.csv')